# Credit Risk Prediction and Explainable Default Analytics
## Phase 2: Exploratory Data Analysis (EDA) & Statistical Profiling

**Objective**: Analyze the UCI *Default of Credit Card Clients* dataset to uncover empirical distributions, credit risk dynamics, delinquency patterns, and multicollinearity ahead of predictive modeling.

> **Task Specification**: This is a *next-month default prediction task* for existing cardholders (`default_payment_next_month`), not an initial loan underwriting application.

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in sys.path
sys.path.append(os.path.abspath(os.path.join("..", "src")))
from data_loader import load_raw_dataset, validate_and_clean_data, DATA_DICTIONARY

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: "%.3f" % x)

### 1. Data Ingestion & Schema Inspection

In [ ]:
df_raw = load_raw_dataset()
df_clean, validation_summary = validate_and_clean_data(df_raw)

print(f"Cleaned dataset shape: {df_clean.shape}")
print("Validation Summary:")
print(json.dumps(validation_summary, indent=2))
df_clean.head()

### 2. Target Class Distribution & Imbalance Analysis

We examine the prevalence of defaults in October 2005 (`default_payment_next_month`).

In [ ]:
target_dist = df_clean["default_payment_next_month"].value_counts()
target_pct = df_clean["default_payment_next_month"].value_counts(normalize=True) * 100

summary_df = pd.DataFrame({"Count": target_dist, "Percentage (%)": target_pct})
display(summary_df)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Non-Default (0)", "Default (1)"], target_dist.values, color=["#2b5c8f", "#d9534f"], width=0.4)
for i, (c, p) in enumerate(zip(target_dist.values, target_pct.values)):
    ax.text(i, c + 500, f"{c:,}\n({p:.1f}%)", ha="center", fontweight="bold")
ax.set_ylabel("Number of Customers")
ax.set_title("Target Distribution (Default vs Non-Default)")
plt.show()

**Interpretation**:
- Non-default cardholders comprise **77.88%** (23,364 cases) while defaults account for **22.12%** (6,636 cases).
- This presents a **moderate class imbalance** (~3.5:1 ratio). Plain classification accuracy is a misleading metric (a naive model predicting all zeros yields 77.88% accuracy).
- Appropriate modeling strategies require stratified splitting, PR-AUC / ROC-AUC optimization, class weighting, and cost-sensitive threshold selection.

### 3. Demographic Features & Group Risk Disparities

We investigate historical default rates across demographic segments (`SEX`, `EDUCATION`, `MARRIAGE`, `AGE`).

> **Caution on Demographics and Ethics**: Demographic associations reflect historical correlations and systemic factors, *not* intrinsic creditworthiness or causation. In production credit risk systems, regulatory frameworks (e.g. ECOA / FCRA) strictly govern or prohibit the use of protected demographic attributes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Gender
sex_map = {1: "Male", 2: "Female"}
sex_grp = df_clean.groupby("SEX")["default_payment_next_month"].mean() * 100
axes[0].bar([sex_map[k] for k in sex_grp.index], sex_grp.values, color="#4682B4", width=0.4)
axes[0].set_title("Default Rate by Sex")
axes[0].set_ylabel("Default Rate (%)")

# Education
edu_map = {1: "Grad School", 2: "University", 3: "High School", 4: "Others/Unk"}
edu_grp = df_clean.groupby("EDUCATION")["default_payment_next_month"].mean() * 100
axes[1].bar([edu_map[k] for k in edu_grp.index], edu_grp.values, color="#5F9EA0", width=0.4)
axes[1].set_title("Default Rate by Education")
axes[1].tick_params(axis="x", rotation=15)

# Marriage
mar_map = {1: "Married", 2: "Single", 3: "Others"}
mar_grp = df_clean.groupby("MARRIAGE")["default_payment_next_month"].mean() * 100
axes[2].bar([mar_map[k] for k in mar_grp.index], mar_grp.values, color="#6495ED", width=0.4)
axes[2].set_title("Default Rate by Marital Status")

plt.tight_layout()
plt.show()

**Interpretation**:
- **Gender**: Males exhibit a slightly higher default rate (24.2%) compared to females (20.8%).
- **Education**: Cardholders with University (23.7%) and High School (25.2%) education have higher default rates than Graduate School holders (19.2%).
- **Marital Status**: Married cardholders show a slightly higher default rate (23.5%) than single cardholders (20.9%).
- **Fairness Implication**: These features should be scrutinized during explainability to ensure the model relies primarily on repayment behavior and liquidity indicators rather than demographic bias.

### 4. Credit Limit and Age Dynamics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Credit limit by status
sns.boxplot(x="default_payment_next_month", y="LIMIT_BAL", data=df_clean, ax=axes[0],
            palette=["#2b5c8f", "#d9534f"], hue="default_payment_next_month", legend=False, width=0.4)
axes[0].set_xticklabels(["Non-Default (0)", "Default (1)"])
axes[0].set_ylabel("Credit Limit (NT$)")
axes[0].set_title("Credit Limit by Default Status")

# Age cohort risk
df_clean["Age_Bracket"] = pd.cut(df_clean["AGE"], bins=[18, 25, 30, 40, 50, 60, 100],
                                 labels=["18-25", "26-30", "31-40", "41-50", "51-60", "60+"])
age_risk = df_clean.groupby("Age_Bracket", observed=False)["default_payment_next_month"].mean() * 100
axes[1].bar(age_risk.index.astype(str), age_risk.values, color="#d9534f", width=0.4)
axes[1].set_ylabel("Default Rate (%)")
axes[1].set_title("Default Rate Across Age Cohorts")
for i, v in enumerate(age_risk.values):
    axes[1].text(i, v + 0.5, f"{v:.1f}%", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

**Interpretation**:
- **Credit Limit**: Defaulters have a noticeably lower median credit limit (NT$ 90,000) than non-defaulters (NT$ 150,000). Lower credit limits reflect prior risk evaluations by the issuing bank.
- **Age**: Default rates follow a mild U-shaped distribution: younger cardholders (18-25: ~27.1%) and older cardholders (60+: ~27.3%) experience higher default rates than cardholders in their thirties (31-40: ~20.4%).

### 5. Repayment Status Dynamics (`PAY_0` to `PAY_6`)

The repayment status features represent the number of months a payment was delayed (-2=no usage, -1=paid in full, 0=revolving balance, 1..9=months delayed).

In [ ]:
pay_status_risk = df_clean.groupby("PAY_0")["default_payment_next_month"].agg(["count", "mean"]).reset_index()
pay_status_risk["default_rate_pct"] = pay_status_risk["mean"] * 100
display(pay_status_risk)

fig, ax = plt.subplots(figsize=(8, 4))
valid_statuses = pay_status_risk[pay_status_risk["count"] >= 20]
ax.plot(valid_statuses["PAY_0"], valid_statuses["default_rate_pct"], marker="o", color="#d9534f", linewidth=2)
ax.set_title("Default Rate vs Most Recent Repayment Status (PAY_0)")
ax.set_xlabel("PAY_0 Status Code (-2: Inactive, -1: Full Pay, 0: Revolving, 1: 1-mo delay, 2: 2-mo delay)")
ax.set_ylabel("Default Rate (%)")
ax.grid(True)
plt.show()

**Interpretation**:
- `PAY_0` is the single strongest univariate predictor of next-month default:
  - Paid in full (`-1`) or no balance (`-2`): ~13-16% default rate.
  - Revolving credit (`0`): ~12.8% default rate.
  - 1-month delay (`1`): ~34.0% default rate.
  - 2-month delay (`2`): ~69.1% default rate.
  - 3+ months delay: >75% default rate.
- A customer transitioning from 1 to 2 months of delinquency represents a severe credit degradation signal.

### 6. Bill Amounts, Payments, and Multicollinearity

In [ ]:
bill_cols = [f"BILL_AMT{i}" for i in range(1, 7)]
pay_cols = [f"PAY_AMT{i}" for i in range(1, 7)]

print("Correlation among 6-month bill statements:")
display(df_clean[bill_cols].corr())

print("Correlation among 6-month payment amounts:")
display(df_clean[pay_cols].corr())

**Interpretation & Multicollinearity Findings**:
- Bill statements across consecutive months (`BILL_AMT1` to `BILL_AMT6`) have pairwise Pearson correlation coefficients between **0.80 and 0.95**, exhibiting high multicollinearity.
- Raw bill amounts also contain negative values (refunds / credit balances) and significant right-skew.
- **Feature Engineering Strategy**:
  1. Compute aggregated metrics (`AVG_BILL_AMT`, `AVG_PAY_AMT`, `MAX_BILL_AMT`).
  2. Compute **Credit Utilization**: `BILL_AMT / LIMIT_BAL`.
  3. Compute **Payment-to-Bill Ratios**: `PAY_AMT / BILL_AMT` with zero/negative bill handling and clamping.
  4. Track delinquency trajectory: count of delayed months (`TOTAL_DELAYS`) and trend (`PAY_0 - PAY_6`).